# Transformer Attention

“小冬瓜有两把刷子”，中的“刷子”的含义，需要分析上下文语境，才能识别其具体的语义。

这个语义仍有模糊的地方，它不是绝对的表示实体的“刷子”或者“能力”。 “一语双冠”正是自然语言分析的复杂之处

语言模型正是要建模出这种模糊的语义。

## 序列建模

word2vec 通过假设窗口局部词汇的关联性，从而构建无监督学习任务来学习 token 的表示。

给定 “刷子” token

- text1: “小冬瓜有两把**刷子**”
- text2: “它用**刷子**画了一只小猫”

在两种语境中， `刷子` 有不同的语义

我们从 embedding 出发, 两句话的 `刷子` 的表示 $E_{刷子}$ 是一样的。

而我们需要通过 **变换** $\mathcal{F}(\cdot):\mathbb{R}^d \rightarrow \mathbb{R}^d$ 实现:

\begin{align}
\mathcal{F}(E_{刷子}) &\rightarrow X_{text1:刷子},\\
\mathcal{F}(E_{刷子}) &\rightarrow X_{text2:刷子},\\
X_\text{text1:刷子} &\neq X_\text{text2:刷子}
\end{align}

In [5]:
import torch 
import torch.nn as nn
import torch.nn.functional as F

vocab_size = 10
seq_len = 8
batch_size = 2
dim = 4


text1 = torch.randint(0, vocab_size - 1, (1, seq_len))[0]
text2 = torch.randint(0, vocab_size - 1, (1, seq_len))[0]
print(text1)
print(text2)

tensor([4, 8, 7, 7, 5, 0, 3, 5])
tensor([2, 4, 2, 2, 3, 5, 3, 4])


In [14]:
idx = 0
text1[idx] = 9
text2[idx] = 9
print(text1)
print(text2)

tensor([9, 8, 7, 7, 5, 9, 3, 5])
tensor([9, 4, 2, 2, 3, 9, 3, 4])


In [15]:
E = nn.Embedding(vocab_size, dim)

In [16]:
seq_embd_1 = E(text1)
seq_embd_2 = E(text2)

print(seq_embd_1[0])
print(seq_embd_2[0])

tensor([ 1.8955,  0.9140, -1.1135, -1.1162], grad_fn=<SelectBackward0>)
tensor([ 1.8955,  0.9140, -1.1135, -1.1162], grad_fn=<SelectBackward0>)


In [26]:
def operation(X):
    X_global = X.mean(dim = 0)
    X = X_global + X
    return X

f1 = operation( seq_embd_1 )[ idx ]
f2 = operation( seq_embd_2 )[ idx ]
print( f1 )
print( f2 )

tensor([ 1.9645,  0.2137, -1.5654, -1.6083], grad_fn=<SelectBackward0>)
tensor([ 1.6893, -0.1191, -1.8284, -1.6586], grad_fn=<SelectBackward0>)


以上，对于位置 $t=0$ 的 context 融合后的特征，是有差别的。

从context-level特征视角：`X = X_global + X` ,  而 X_global 为 序列性的整体表征

词的语义受上下文影响。

序列建模的目标是：找到一种能够高效组合全局信息的方式，来表示词元在语境中的语义

## 高效序列建模

对于文本中 “小冬瓜有两把刷子“， `刷子` 的语义是根据 “语法” 关系表示出来的。

- 语法规则：我们可以通过提取主谓宾定等词性，并分析词与词（或整体）之间的联系，从而表示 `刷子`含义
- 自动化规则：语法规则在于它是复杂的，比如“定语重置”或“倒装句”，中英文语法关系也有区别，期望能够**自动化** 表示 token 在  context 语义：

在复杂的语言模式中，通用性的语言表示，实际上是要自动化的刻画 token 在 context 中的表示。

我们定义权重，来刻画词元之间的语法关系

|                | 小   | 冬   | 瓜   | 有   | 两   | 把   | 刷   | 子   |
| -------------- | ---- | ---- | ---- | ---- | ---- | ---- | ---- | ---- |
| $\overline{w}$ | 1/8  | 1/8  | 1/8  | 1/8  | 1/8  | 1/8  | 1/8  | 1/8  |
| $w$            | 0.1  | 0.1  | 0.5  | 0.01 | 0.01 | 0.08 | 0.1  | 0.1  |
| $w_刷$         | 0.0  | 0.1  | 0.1  | 0.4  | 0.0  | 0.0  | 0.3  | 0.1  |

- $\overline{w}$：每个 token 对全局特征的 重要性 相同
- $w$: 每个 token 对全局特征 重要性 不同。
- $w_刷$: `刷` token 视角，它与其他词元之间的关系。其精细化的表示：一个特异的语法规则联系。 对于其他token也有独立的权重来表示自动化语法规则

再者，对于较长的小说，也许只有关键的情节才能推动故事的发展，即 故事 与 情节有强关联，与其他描写无关或关联小。


|                | 小   | 冬   | 瓜   | 有   | 两   | 把   | 刷   | 子   |
| -------------- | ---- | ---- | ---- | ---- | ---- | ---- | ---- | ---- |
| $w_刷$         | 0.0  | 0.0  | 0.0  | 0.0  | 0.2  | 0.4  | 0.3  | 0.1  |

如果我们刻画了一个错误的权重, `刷子` 就难以表示在 context 中的语义。

$$
S_i = \sum_j w_{ij} X_j
$$

所以难点其实在于 如何建模 “自动化语法规则” 的权重$w_i$，从而提高语义理解。

如下我们定义向量内积，来计算特征向量之间的关联程度。（内积，只是度量相关性的一种实现）

$$
w_{ij} = X_i X_j^T
$$

In [46]:
X = seq_embd_1
X_0 = seq_embd_1[i,:].unsqueeze(dim = 0)
print(X.shape)
print(X_0.shape)

## 循环实现
i = 0
X_weight_i = torch.zeros(1, dim)
w_ij = torch.zeros(seq_len)
for j in range(seq_len):
    w_ij[j] = X[i,:] @ X[j,:].t()
    X_weight_i += w_ij[j] * X[j,:] # weight * feature
print(w_ij/w_ij.max())
print(X_weight_i)


# 矩阵操作实现
s = X_0 @ X.t()
print(s.shape)
X_weight_i = s @ X
print(X_weight_i)

torch.Size([8, 4])
torch.Size([1, 4])
tensor([ 1.0000, -0.2222,  0.3532,  0.3532, -0.6241,  1.0000, -0.6076, -0.6241],
       grad_fn=<DivBackward0>)
tensor([[ 51.0115,  40.7847, -23.9816,  -3.6762]], grad_fn=<AddBackward0>)
torch.Size([1, 8])
tensor([[ 51.0115,  40.7847, -23.9816,  -3.6762]], grad_fn=<MmBackward0>)


## 注意力机制

上述，已经找出变换 $\mathcal{F}(\cdot):\mathbb{R}^d \rightarrow \mathbb{R}^d$

本质上，我们是在找 “词在序列中的表征”， 

\begin{align}
S_i = \sum_j w_{ij} X_j,\\
w_{ij} = X_i X_j^T
\end{align}

展开式子

\begin{align}
S_i = w_{i1} X_1 + w_{i2} X_2 + \ldots + w_{iN} X_N 
\end{align}

其中有两部分表示 $w_{ij}$ 权重项用于 衡量各 token 的贡献，$X_i$ 即是常规的特征表示。

进一步展开

\begin{align}
S_i = (X_i X_1^T)\cdot X_1 + (X_i X_2^T)\cdot X_2 + \ldots + (X_i X_N^T)\cdot X_N, (X_i X_N^T)\in\mathbb{R}
\end{align}

其中，(X_i X_N^T) 是标量。我们将每一项都进行线性特征变换


| $(X_i$    | $ X_j^T )$    | $\cdot$ | $X_j$    |
| --------- | ------------- | ------- | -------- |
| $(X_iW_q$ | $(X_jW_k)^T)$ | $\cdot$ | $X_jW_v$ |
| $(Q_i$    | $K^T_j)$      | $\cdot$ | $V_j$    |
| 查询      | 键            | $\cdot$ | 值       |

- query：查询向量，即 词元$i$ 想要知道，他在这个 context 中的表示。即查询词元 $i$ 拿了一把钥匙
- key：键向量，即 context 中的每个词元 就是一个门，所以两个词元之间的相关程度，需要 查询词元拿着钥匙$Q_i$ **访问** 每一道门$K_j$
- value：值向量， 即是门里面内容$V_j$

即一个查询词元$Q_i$，敲开所有的键门$K_j$并访问了内容$V_j$, 综合了所有信息后才有 **融合** 的表示

\begin{align}
S_i &= \sum_j (Q_iK^T_j) V_j,\\
Q_i &= X_iW_Q \\
K_j &= X_jW_K \\
V_j &= X_jW_V \\
\end{align}

其中 $W_Q,W_K,W_V \in \mathbb{R}^{d \times}$ , 对各个词特征进行投影变换，实现查询、键、值的表示功能。上式即为**注意力机制**，其计算输出称之为 **注意力特征**

那么，原始的计算形式**特征$(x_{i})$加权$(w_{ij})$组合$(\sum)$**，同样是**注意力机制**， 

\begin{align}
S_i = \sum_j w_{ij} X_j,\\
\end{align}

**既然原始的输入向量能够做 注意力计算，为什么还需要投影？**

考虑$w_{ij}$ 为 $w_{刷:}$

|                | 小   | 冬   | 瓜   | 有   | 两   | 把   | 刷   | 子   |
| -------------- | ---- | ---- | ---- | ---- | ---- | ---- | ---- | ---- |
| $w_刷$         | 0.0  | 0.0  | 0.0  | 0.0  | 0.2  | 0.4  | 0.3  | 0.1  |

$w_\text{刷,瓜} = Q_\text{刷}K_\text{瓜} = 0.0 $ 由于这种表示，忽略了 主语“小冬瓜”， 那么就会造成`刷`的语义错误，使得任务出错.

反向传播时调整$W_Q',W_K'$, 从而调整权重 $w_\text{刷,瓜} = Q_\text{刷}'K_\text{瓜}' = 0.2 $，改变注意力特征关系，而$W_v'$ 同理。

引入参数对表征进行投影，实际是希望投影后的特征，**让查询能够高效注意到对预测任务有重要贡献的词元**。



In [56]:
class AttentionSimplest(nn.Module):
    def __init__(self, dim_in, dim_out):
        super().__init__()
        self.WQ = nn.Linear(dim_in, dim_out)
        self.WK = nn.Linear(dim_in, dim_out)
        self.WV = nn.Linear(dim_in, dim_out)
        self.WO = nn.Linear(dim_in, dim_out) # W_O 对输出做一层投影，对齐到外部空间
        
    def forward(self, X, i):
        seq_len, dim = X.shape
        query_i = X[i, :].unsqueeze(dim = 0)
        qi = self.WQ(query_i)
        K = self.WK(X)
        V = self.WV(X)

        attn_i = torch.zeros(1, dim)
        for j in range(seq_len):
            w_ij = qi @ K[j,:].t()
            attn_i += w_ij * V[j,:]

        output = self.WO(attn_i)
        return output

    def forward_basic(self, X, i):
        """
        此版本不做投影变换，仍然能算注意力
        """
        seq_len, dim = X.shape
        qi = X[i, :].unsqueeze(dim = 0)
        K = X
        V = X
        attn_i = torch.zeros(1, dim)
        for j in range(seq_len):
            w_ij = qi @ K[j,:].t()
            attn_i += w_ij * V[j,:]
        return attn_i

X = torch.randn(seq_len, dim)
attn = AttentionSimplest(dim, dim)


# 我们对 每个查询, 都能获得 token_i 在 context 中的语义表示
O_0 = attn(X, 0)
O_1 = attn(X, 1)
print(O_0)
print(O_1)


# 我们对 每个查询, 不经过投影变换
O_0 = attn.forward_basic(X, 0)
O_1 = attn.forward_basic(X, 1)
print(O_0)
print(O_1)

# 其输出，不带 grad_fn.

tensor([[ 1.5075, -0.5229,  0.2631,  0.7690]], grad_fn=<AddmmBackward0>)
tensor([[-2.3856,  2.1373,  0.6376, -0.8951]], grad_fn=<AddmmBackward0>)
tensor([[-2.2240,  8.7158, -8.1958, -0.4844]])
tensor([[  6.8391, -15.9405,  21.8838,  12.9560]])


## 注意力分数归一化

In [62]:
Q_i = torch.randn(1, dim)
K = torch.randn(seq_len, dim)
V = torch.randn(seq_len, dim)

S = Q_i @ K.t()
print(S)

# 归一化处理
P = F.softmax(S, dim = -1)
print(P)
print(P.sum())

O = PV


# 归一化处理 softmax
def softmax(X):
    m = torch.max(X)
    X_exp = torch.exp(X - m)
    L = torch.sum(X_exp)
    P = X_exp / L
    return P

P = softmax(S[0,:])
print(P)
print(P.sum())

tensor([[-0.7932,  0.6398, -3.2274, -2.2778, -1.0054, -2.3449,  3.4850,  0.6394]])
tensor([[0.0121, 0.0506, 0.0011, 0.0027, 0.0098, 0.0026, 0.8706, 0.0506]])
tensor(1.)
tensor([0.0121, 0.0506, 0.0011, 0.0027, 0.0098, 0.0026, 0.8706, 0.0506])
tensor(1.)


## 掩码注意力

我们在之前的例子中，有注意力分数：

|                | 小   | 冬   | 瓜   | 有   | 两   | 把   | 刷   | 子   |
| -------------- | ---- | ---- | ---- | ---- | ---- | ---- | ---- | ---- |
| $w_刷$         | 0.0  | 0.1  | 0.1  | 0.4  | 0.0  | 0.0  | 0.3  | 0.1  |

如果我们人为定义规则，要求不能访问“奇数位置”的键门，即对“K_j, j%2 = 1” 的门进行上锁。

|                | 小   | 冬   | 瓜   | 有   | 两   | 把   | 刷   | 子   |
| -------------- | ---- | ---- | ---- | ---- | ---- | ---- | ---- | ---- |
| $w_刷$         | 0.0  | 0.1  | 0.1  | 0.4  | 0.0  | 0.0  | 0.3  | 0.1  |
| $\text{mask}_刷$         | 1  | 0  | 1  | 0 | 1  | 0  | 1  | 0  |

如果不能开键（key）门, 那么也不能访问内容（value）, 同理我们对每个 token 都有独立的 $\text{mask}_i$ 锁


$$
\begin{align}
S_i = \sum_j \textcolor{red}{\text{mask}_{ij}} w_{ij} X_j,\\
w_{ij} = X_i X_j^T
\end{align}
$$

In [69]:
mask = torch.zeros(1,seq_len)
idx = torch.arange(1, seq_len, 2)
print(idx)
mask[0, idx] = 1
print(mask)

tensor([1, 3, 5, 7])
tensor([[0., 1., 0., 1., 0., 1., 0., 1.]])


In [79]:
# 掩码注意力
Q_i = torch.randn(1, dim)
K = torch.randn(seq_len, dim)
V = torch.randn(seq_len, dim)

S = Q_i @ K.t()
print(S)

# 增加mask
S_mask = S * mask
print(S)

# 归一化处理
P = F.softmax(S_mask, dim = -1)
print(P) # 有问题, mask 掉的 token 仍有 访问权重
print(P.sum())

tensor([[-0.8568,  1.3915, -0.8499, -2.6875,  0.2577, -1.2579,  2.8600, -0.2869]])
tensor([[-0.8568,  1.3915, -0.8499, -2.6875,  0.2577, -1.2579,  2.8600, -0.2869]])
tensor([[0.1096, 0.4407, 0.1096, 0.0075, 0.1096, 0.0312, 0.1096, 0.0823]])
tensor(1.0000)


In [86]:
idx = torch.arange(0, seq_len, 2)
print(idx)

# 增加mask
S_inf_mask = S.clone()
S_inf_mask[0,idx] = -10000.0 # 在 mask 掉的分数置为 负无穷($-\infty$)
print(S_inf_mask)

# 归一化处理
P = F.softmax(S_inf_mask, dim = -1)
print(P) # 没问题
print(P.sum())

tensor([0, 2, 4, 6])
tensor([[-1.0000e+04,  1.3915e+00, -1.0000e+04, -2.6875e+00, -1.0000e+04,
         -1.2579e+00, -1.0000e+04, -2.8691e-01]])
tensor([[0.0000, 0.7848, 0.0000, 0.0133, 0.0000, 0.0555, 0.0000, 0.1465]])
tensor(1.)


In [93]:
a = torch.tril(torch.ones(1,5,5))
torch.where(a==0)

(tensor([0, 0, 0, 0, 0, 0, 0, 0, 0, 0]),
 tensor([0, 0, 0, 0, 1, 1, 1, 2, 2, 3]),
 tensor([1, 2, 3, 4, 2, 3, 4, 3, 4, 4]))

In [98]:
## 完整注意力
import math

class ScaleDotProductAttention(nn.Module):
    def __init__(self, dim_in, dim_out):
        super().__init__()
        self.WQ = nn.Linear(dim_in, dim_out)
        self.WK = nn.Linear(dim_in, dim_out)
        self.WV = nn.Linear(dim_in, dim_out)
        self.WO = nn.Linear(dim_in, dim_out) 
        
    def forward(self, X, mask):
        batch_size, seq_len, dim = X.shape
        Q = self.WQ(X)
        K = self.WK(X)
        V = self.WV(X)

        # 多个 q_i 计算注意力特征
        S = Q @ K.transpose(1,2) / math.sqrt(dim) # 1. 为什么要除于 \sqrt{d}

        idx =torch.where(mask==0)
        S[idx[0],idx[1],idx[2]] = -10000.0
        
        P = torch.softmax(S, dim = -1) # 行 softmax
        Z = P @ V
        output = self.WO(Z)
        
        return output

X = torch.randn(64, seq_len, dim)
mask = torch.tril(torch.ones(64, seq_len, dim))
model = ScaleDotProductAttention(dim, dim)
Y = model(X, mask)
print(Y.shape)

torch.Size([64, 8, 4])
